# 05 — Deploy an App from a Dockerfile

When you want to customise a container, you write a `Dockerfile` that describes your changes.
OpenShift can build the image for you and then deploy it.

We will add a config file to the welcome app and mount it into the container using a ConfigMap.

Run each cell from top to bottom by pressing **Shift + Enter**.

In [ ]:
import os
os.environ["UV_EXTRA_INDEX_URL"] = "https://pypi.org/simple"

# enable colour output for all print() calls in this notebook
!uv pip install rich -q
from rich import print, print_json

## Step 1 — Create a working directory and your files

In [ ]:
!mkdir -p foo

In [ ]:
%%writefile foo/foobar.properties
a=b
c=100

In [ ]:
%%writefile foo/Dockerfile
FROM quay.io/eformat/welcome
COPY foobar.properties /tmp

## Step 2 — Build the image on OpenShift

This tells OpenShift to compile your Dockerfile into a container image.

In [ ]:
PROJECT = !oc project -q
PROJECT = PROJECT[0].strip()
print("Project:", PROJECT)

In [ ]:
!oc new-build -n {PROJECT} --name=welcome --strategy=docker --binary

In [ ]:
!oc start-build -n {PROJECT} welcome --from-dir=foo --follow

## Step 3 — Deploy and create a public route

In [ ]:
!oc new-app welcome

In [ ]:
!oc create route edge --service=welcome --port=8080 --insecure-policy=Redirect

## Step 4 — Add a ConfigMap

A ConfigMap stores configuration data separately from the container image.
We will create one and mount it into the running pod.

In [ ]:
%%writefile configmap.yaml
kind: ConfigMap
apiVersion: v1
metadata:
  name: welcome
data:
  application.properties: |-
    helloservice:
      message: hello, world !

In [ ]:
!oc -n {PROJECT} create -f configmap.yaml

In [ ]:
!oc -n {PROJECT} set volume dc/welcome \
    --add --overwrite \
    --name=welcomeconfigmap \
    -m /tmp/application.properties \
    --sub-path=application.properties \
    -t configmap \
    --configmap-name=welcome

## Step 5 — Verify the files inside the pod

Wait for the pod to restart (about 30 seconds), then check both files are present.

In [ ]:
POD = !oc -n {PROJECT} get pods -l app=welcome --template='{{range .items}}{{.metadata.name}}{{end}}'
POD = POD[0].strip()
print("Pod name:", POD)

In [ ]:
print("--- application.properties (from ConfigMap) ---")
!oc -n {PROJECT} exec {POD} -- cat /tmp/application.properties

print()
print("--- foobar.properties (baked into image) ---")
!oc -n {PROJECT} exec {POD} -- cat /tmp/foobar.properties